# Existence equation — panel logit

Whether a cross-pool arbitrage exists. Here the **onset** risk set (`gap_lag == 0`): given no gap
is open, does one appear at `t`?

$$\Pr\!\big(D_{p,t}=1 \mid X_{p,t-1}\big)=\Lambda(\eta_{p,t}),\qquad \Lambda(z)=\frac{1}{1+e^{-z}}$$

$$
\eta_{p,t}=\alpha_p
+\beta_2\,\log(\text{base\_fee}_t)+\beta_3\,\text{gas\_util}_{t-1}+\beta_4\,\log(1+\text{tip\_p90}_{t-1})
+\beta_5\,\overline{\log(1+\text{mev})}_{p,t-1}+\beta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}
+\beta_7\,\log(\text{ewma\_vol}_t)
+\gamma_{h(t)}
$$

`base_fee` and `ewma_vol` enter at `t`; every other regressor is lagged one block to be
predetermined. $\alpha_p$ are pool-pair fixed effects and $\gamma_{h(t)}$ are hour-of-day fixed
effects (day-of-week and iso-week are omitted — near-collinear and low-signal on a sub-week window).
All logic lives in `arblib.estimation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import estimation as est
from arblib.config import STUDY as S

panel = est.load_panel(S)
print("panel:", panel.shape)

panel: (1162134, 27)


## Onset risk set

Rows with `gap_lag == 0`, dropping pairs with no `D` variation (uninformative under pair FE).

In [2]:
onset = est.build_risk_set(panel, quantile=0.2, condition="onset")
# EPV floor: drop pairs whose rarer outcome class (here usually the onset events D=1) is too thin


onset: (1087557, 15) | D mean: 0.023 | pairs: 27


In [3]:
onset.groupby("pair").size()

pair
pancake_1_vs_pancake_2    38195
uniswap_1_vs_pancake_1    35981
uniswap_1_vs_pancake_2    27251
uniswap_1_vs_uniswap_2    41872
uniswap_1_vs_uniswap_3    33238
uniswap_1_vs_uniswap_4    35695
uniswap_1_vs_uniswap_5    42929
uniswap_1_vs_uniswap_6    42924
uniswap_2_vs_pancake_1    42504
uniswap_2_vs_pancake_2    41225
uniswap_2_vs_uniswap_3    40831
uniswap_2_vs_uniswap_4    39961
uniswap_2_vs_uniswap_5    43021
uniswap_2_vs_uniswap_6    43010
uniswap_3_vs_pancake_1    41398
uniswap_3_vs_pancake_2    34888
uniswap_3_vs_uniswap_4    41620
uniswap_3_vs_uniswap_5    42947
uniswap_3_vs_uniswap_6    42999
uniswap_4_vs_pancake_1    40664
uniswap_4_vs_pancake_2    36629
uniswap_4_vs_uniswap_5    42895
uniswap_4_vs_uniswap_6    42995
uniswap_5_vs_pancake_1    43008
uniswap_5_vs_pancake_2    42915
uniswap_6_vs_pancake_1    43009
uniswap_6_vs_pancake_2    42953
dtype: int64

In [4]:
onset = est.drop_pairs_by_event_floor(onset, S.min_events_per_pair, label="onset")

onset: dropping 8 pairs with < 55 rarer-class events (min(n_D0, n_D1)): ['uniswap_2_vs_uniswap_5', 'uniswap_2_vs_uniswap_6', 'uniswap_3_vs_uniswap_5', 'uniswap_3_vs_uniswap_6', 'uniswap_4_vs_uniswap_6', 'uniswap_5_vs_pancake_1', 'uniswap_6_vs_pancake_1', 'uniswap_6_vs_pancake_2']
onset: kept 19 pairs, 743615 rows


In [5]:
onset["D"].mean()

np.float64(0.03338959004323474)

## Fit — logit & probit

Pool-pair fixed effects `C(pair)`, cluster-robust SEs by pair. Covariates are mean-centred so the
intercept is read at an average observation.

In [6]:
onset_c = est.center_continuous(onset, est.ONSET_TERMS)
res_logit  = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="logit")
res_probit = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="probit")
print(res_logit.summary())
#print(res_probit.summary())

                           Logit Regression Results                           
Dep. Variable:                      D   No. Observations:               743615
Model:                          Logit   Df Residuals:                   743567
Method:                           MLE   Df Model:                           47
Date:                Sat, 22 Aug 2026   Pseudo R-squ.:                  0.2050
Time:                        11:04:40   Log-Likelihood:                -86507.
converged:                       True   LL-Null:                   -1.0882e+05
Covariance Type:              cluster   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -3.2077      0.041    -78.417      0.000      -3.288      -3.128
C(pair)[T.uniswap_1_vs_pancake_1]     0.2545      0.056      4

In [7]:
print(res_probit.summary())

                          Probit Regression Results                           
Dep. Variable:                      D   No. Observations:               743615
Model:                         Probit   Df Residuals:                   743567
Method:                           MLE   Df Model:                           47
Date:                Sat, 22 Aug 2026   Pseudo R-squ.:                  0.2070
Time:                        11:04:41   Log-Likelihood:                -86291.
converged:                       True   LL-Null:                   -1.0882e+05
Covariance Type:              cluster   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -1.7335      0.022    -80.429      0.000      -1.776      -1.691
C(pair)[T.uniswap_1_vs_pancake_1]     0.0965      0.028      3

## Separation diagnostic — where is it concentrated?

If statsmodels flags quasi-separation, this shows *where* it sits: the share of near-perfectly
predicted observations, how they split across `pair` / `hour` cells (a few sparse interaction cells
vs. the whole sample), and the fixed-effect dummies carrying the classic signature — a coefficient
pushed to an extreme value with an unusually tight SE (large `|z|`), or an unidentified `NaN`-SE one.
A concentration in a handful of thin `pair × hour` cells confirms the sparse-cell story (the economic
covariates stay identified); a large share spread everywhere would be a real problem instead.

In [8]:
est.separation_report(res_logit, onset_c)



perfectly predicted (fitted p < 0.0001 or > 0.9999): 0 / 743615 obs (0.0%)

fixed-effect dummies by |z| (separation signature = big |coef|, tiny SE), top 8:
                                    coef  std_err    abs_z
C(pair)[T.uniswap_5_vs_pancake_2] -3.033    0.015  201.606
C(pair)[T.uniswap_2_vs_pancake_1] -2.249    0.028   79.895
C(pair)[T.uniswap_4_vs_uniswap_5] -3.059    0.039   79.300
C(pair)[T.uniswap_3_vs_pancake_1] -1.284    0.017   75.608
C(pair)[T.uniswap_2_vs_uniswap_4] -1.079    0.015   70.230
C(pair)[T.uniswap_2_vs_uniswap_3] -1.213    0.022   55.301
C(pair)[T.uniswap_3_vs_uniswap_4] -1.410    0.027   51.878
C(pair)[T.uniswap_1_vs_uniswap_5] -3.561    0.073   48.454


## Average marginal effects

Logit coefficients are not probability changes, so report AMEs (logit and probit agree closely).

In [9]:
est.average_marginal_effects(res_logit, est.ONSET_TERMS)

,dy/dx,Std. Err.,z,Pr(>|z|),Conf. Int. Low,Cont. Int. Hi.
log_base_fee,0.011553,0.000836,13.812812,2.133375e-43,0.009914,0.013192
gas_util_lag,-0.006533,0.001028,-6.353094,2.110262e-10,-0.008548,-0.004517
tip_p90_lag,0.000860,0.000266,3.231976,1.229373e-03,0.000339,0.001382
mev_lag,0.003674,0.000510,7.201959,5.935374e-13,0.002674,0.004674
freq_lag,0.012744,0.001846,6.904781,5.028089e-12,0.009127,0.016362
log_vol,0.016666,0.002027,8.223460,1.977148e-16,0.012694,0.020638


## Multicollinearity check

VIFs (worry above ~10) and the covariate correlation matrix.

In [10]:
print(est.variance_inflation(onset, est.ONSET_TERMS))
onset[est.ONSET_TERMS].corr().round(3)

const           2217.060550
log_base_fee       1.133048
gas_util_lag       1.055722
tip_p90_lag        1.117082
mev_lag            1.238631
freq_lag           1.166418
log_vol            1.391198
Name: VIF, dtype: float64


,log_base_fee,gas_util_lag,tip_p90_lag,mev_lag,freq_lag,log_vol
log_base_fee,1.000,0.037,0.018,0.122,0.061,0.335
gas_util_lag,0.037,1.000,-0.220,0.005,0.022,-0.002
tip_p90_lag,0.018,-0.220,1.000,0.138,0.121,0.219
mev_lag,0.122,0.005,0.138,1.000,0.322,0.376
freq_lag,0.061,0.022,0.121,0.322,1.000,0.293
log_vol,0.335,-0.002,0.219,0.376,0.293,1.000
